# Timings of application

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import gc
import numpy as np
import pickle
from time import time
import matplotlib.pyplot as plt
from figure_manager import FigureManager
from plots import *

import DynamicTimeAllocationModel

path = 'output/'
if not os.path.exists(path):
    os.makedirs(path)
    
plt.rcParams.update({'font.size': 14})
fm = FigureManager(path, use_latex=False)

# c++ settings
do_compile = True
load_par = True
threads = 64

from EconModel import cpptools
cpptools.setup_nlopt(folder='cppfuncs/', do_print=True)

NLopt already installed


## Monte Carlo Settings

In [2]:
MC_num = 1 #200 # number of Monte Carlo simulations
C_num_grid = (25,50,75,100,200) # number of grid points in consumption grid i iEGM
do_print = True # whether to print results during Monte Carlo iterations

## Illustrate the pre-computation of consumption interpolator

In [3]:
filename = 'calibrated_par'
with open(f"par_files/{filename}.pkl", "rb") as f:
    settings = pickle.load(f)

## Solve the "true" model using many grid points in VFI

In [4]:
# overwrite settings with specs_true
settings_true = settings.copy()

settings_true['do_multistart'] = False
settings_true['do_egm'] = False
settings_true['interp_method'] = 'linear'
settings_true['interp_inverse'] = True
settings_true['precompute_intratemporal'] = True
settings_true['centered_gradient'] = True

settings_true['num_A'] = 50
settings_true['num_A_pd'] = 50
settings_true['num_K'] = 12
settings_true['num_power'] = 15
settings_true['num_love'] = 15
settings_true['num_types'] = 4
settings_true['num_Ctot'] = 400
settings_true['num_marg_u'] = 400

model_true = DynamicTimeAllocationModel.HouseholdModelClass(par=settings_true)
model_true.link_to_cpp(force_compile=do_compile)

In [5]:
%time model_true.solve()
%time model_true.simulate()

CPU times: total: 11d 17h 8min 32s
Wall time: 4h 58min 13s
CPU times: total: 34.5 s
Wall time: 2 s


Print lifetime utility of the "true" model

In [6]:
print(model_true.sim.mean_lifetime_util[0]) # 158.2609297004482

158.89345216058354


Print the fraction of points where power is updated in the simulation

In [7]:
power_diff = np.ones(model_true.sim.power.shape) + np.nan
power_diff[:,1:] = np.diff(model_true.sim.power, axis=1)
power_updated = (np.isclose(power_diff, 0.0, atol=1e-5) == False)
power_updated[np.isnan(power_diff)] = False
print(power_updated.sum()/model_true.sim.couple.sum()*100, '%') # 0.5794772070436305 %

0.5778261005909712 %


## Accuracy measures

In [8]:
model_specs = {    
    'iegm': {'table_name':'iEGM', 'do_egm': True, 'interp_method': 'linear', 'do_multistart': False},
    'iegm (num)': {'table_name':'EGM (num)', 'do_egm': True, 'interp_method': 'numerical', 'do_multistart': False},
    'vfi (no multistart)': {'table_name':'VFI (no multistart)', 'do_egm': False, 'do_multistart': False},
    'vfi': {'table_name':'VFI', 'do_egm': False, 'do_multistart': True},
}


In [11]:
models = {}
results = {}
for name_base,specs in model_specs.items():    
    
    if ('iegm' in name_base) and (specs['interp_method']!='numerical'):
        C_num_grid_now = C_num_grid
    else:
        C_num_grid_now = (20,) # not used
    
    for num_C in C_num_grid_now:  
        name = name_base + (f' num_C={num_C}' if len(C_num_grid_now)>1 else '')
        key = num_C if len(C_num_grid_now)>1 else ''
        
        if name not in results:
            results[name] = {}
        results[name][key] = {
            'timing':np.nan+np.ones(MC_num),
            'timing_rel':np.nan+np.ones(MC_num),
            'MSE':np.nan+np.ones(MC_num),
            'l_MAD':np.nan+np.ones(MC_num),
            'power_MAD':np.nan+np.ones(MC_num),
            'divorce_MAD':np.nan+np.ones(MC_num),
            'C_MAD':np.nan+np.ones(MC_num),
            'util_diff_pct':np.nan+np.ones(MC_num),
            'comp':np.nan+np.ones(MC_num),
            'lifetime_util':np.nan+np.ones(MC_num),
        } 
        
        if do_print: print(name,key)
        model_name = name
        
        # setup model and solution containers
        models[model_name] = DynamicTimeAllocationModel.HouseholdModelClass(par={**settings,**specs,'num_Ctot':num_C, 'num_marg_u': num_C})
        
        # loop over Monte Carlo iterations
        for i_mc in range(MC_num):
            if do_print: print(f'{i_mc+1}/{MC_num} running...')
            
            # re-simulate true model for this Monte Carlo iteration to get new draws
            true = model_true
            
            true.par.seed = i_mc
            true.draw_shocks_and_initial_states()
            true.link_to_cpp(force_compile=False)
            true.simulate()
            
            # solution time
            models[model_name].link_to_cpp(force_compile=False)
            t0 = time()
            models[model_name].solve()
            secs = time() - t0
            if do_print: print(f'Solution time: {secs:.2f} seconds')
            
            # accuracy
            models[model_name].par.seed = i_mc
            models[model_name].draw_shocks_and_initial_states()
            models[model_name].simulate()

            l_MAD, divorce_MAD, power_MAD,C_MAD = models[model_name].measure_accuracy(true)
            if do_print: print(f'MAD in labor function: {l_MAD:.4f}')
            if do_print: print(f'MAD in divorce: {divorce_MAD:.4f}')
            if do_print: print(f'MAD in updated power: {power_MAD:.4f}')
            if do_print: print(f'MAD in consumption function: {C_MAD:.4f}')
            
            lifetime_util = models[model_name].sim.mean_lifetime_util[0]
            if do_print: print(f'Lifetime utility: {lifetime_util:.12f}')
            
            comp = models[model_name].wealth_compensation(true)
            if do_print: print(f'Wealth compensation (pct of household exp. income): {comp:.4f}', end='\n\n')
            
            
            results[name][key]['timing'][i_mc] = secs
            results[name][key]['l_MAD'][i_mc] = l_MAD
            results[name][key]['divorce_MAD'][i_mc] = divorce_MAD
            results[name][key]['power_MAD'][i_mc] = power_MAD
            results[name][key]['C_MAD'][i_mc] = C_MAD
            results[name][key]['lifetime_util'][i_mc] = lifetime_util
            results[name][key]['comp'][i_mc] = comp
            
            # delete to free memory
            del models[model_name]
            gc.collect()

iegm num_C=25 25
1/1 running...
Solution time: 156.27 seconds
MAD in labor function: 0.1058
MAD in divorce: 0.0000
MAD in updated power: 0.0189
MAD in consumption function: 2.3208
Lifetime utility: 138.620271093487
Wealth compensation (pct of household exp. income): 168.2432

iegm num_C=50 50
1/1 running...
Solution time: 154.46 seconds
MAD in labor function: 0.0804
MAD in divorce: 0.0000
MAD in updated power: 0.0191
MAD in consumption function: 0.3801
Lifetime utility: 149.136168985437
Wealth compensation (pct of household exp. income): 57.4521

iegm num_C=75 75
1/1 running...
Solution time: 169.07 seconds
MAD in labor function: 0.0501
MAD in divorce: 0.0000
MAD in updated power: 0.0191
MAD in consumption function: 0.3825
Lifetime utility: 151.542170552481
Wealth compensation (pct of household exp. income): 39.8780

iegm num_C=100 100
1/1 running...
Solution time: 216.94 seconds
MAD in labor function: 0.0022
MAD in divorce: 0.0000
MAD in updated power: 0.0191
MAD in consumption functi

In [12]:
model =  DynamicTimeAllocationModel.HouseholdModelClass(par={**settings,**model_specs['iegm'],'num_Ctot':100, 'num_marg_u': 100})

In [13]:
model.link_to_cpp(force_compile=False)
model.solve()

In [14]:
(model.sol.Cwd_priv_couple_to_couple < 0).sum()

np.int64(3446)

In [15]:
(model_true.sol.Cwd_priv_couple_to_couple + model_true.sol.Cmd_priv_couple_to_couple + model_true.sol.Cd_inter_couple_to_couple < 0).sum()

np.int64(0)

In [16]:
(model_true.sol.Cwd_priv_couple_to_couple < 0).sum()

np.int64(40485)

In [17]:
(model.sol.Cwd_priv_couple_to_couple[slice(None), slice(None), slice(None), slice(None), slice(None), -1] < 0).sum()

np.int64(3446)

In [18]:
model_true.sol.Cwd_priv_couple_to_couple[(model_true.sol.Cwd_priv_couple_to_couple < 0)].mean()

np.float64(-7.564265663491128)

In [19]:
print(results['vfi (no multistart)']['']['lifetime_util'][0])
print(results['vfi']['']['lifetime_util'][0])

134.21450702035892
134.214548117716


In [24]:
# write to latex file
table_path = 'output/MC_limited'
output = ('timing','timing_rel','l_MAD', 'divorce_MAD', 'power_MAD', 'C_MAD','comp')
output_labels = '& time (m) & rel. & Error(l) & Error(divorce) & MAD(power) & MAD(C)  & Comp(A) '

num_cols_per_model = len(output)
    
num_models = 1 
num_cols = 2 + num_cols_per_model*num_models

with open(table_path+'.tex','w') as f:
    # table head
    f.write('\\begin{tabular}{ll' + 'c'* (num_cols-2) + '} \\toprule\n')
    f.write(f'&& \\multicolumn{{{num_cols_per_model}}}{{c}}{{Application model}}  \\\\ \n ')
    f.write(f'\\cmidrule(lr){{3-{2+num_cols_per_model}}} &')
    for _ in range(num_models): f.write(output_labels)
    f.write('\\\\ \\midrule \n')
    
    # table body
    model_specs_now = model_specs
    for name_base,specs in reversed(model_specs_now.items()):
        table_name = specs['table_name']
        
        if ('iegm' in name_base) and (specs['interp_method']!='numerical'):
            C_num_grid_now = C_num_grid
        else:
            C_num_grid_now = (20,) # not used
        
        #num_multi = 1 if len(C_num_grid_now)>1 else 2
        f.write(f'\\multicolumn{{{2}}}{{l}}{{{table_name}}} ')
        for iC,num_C in enumerate(C_num_grid_now):  
            if len(C_num_grid_now)>1:
                if iC==0: # skip entire row
                    f.write(f'\\multicolumn{{{num_cols-2}}}{{c}}{{}} \\\\ \n')
                
                f.write(f'& $\\#_C$={num_C}')
            
            name = name_base + (f' num_C={num_C}' if len(C_num_grid_now)>1 else '')
            key = num_C if len(C_num_grid_now)>1 else ''
            
            for out in output:
                if out == "timing":
                    avg = np.mean(results[name][key][out]) / 60
                elif out == 'timing_rel':
                    avg = np.mean(results[name][key]['timing'])/np.mean(results['vfi']['']['timing'])
                else:
                    avg = np.mean(results[name][key][out])
                if np.isnan(avg):
                    f.write('& N.A. ')
                else:
                    f.write(f'& {avg:.3f} ')
            f.write('\\\\ \n')
    f.write('\\bottomrule \\end{tabular}\n')
    
    
# output the table here by looping through all lines in the file and printing them
with open(table_path+'.tex','r') as f:
    for line in f:
        print(line,end='')  



\begin{tabular}{llccccccc} \toprule
&& \multicolumn{7}{c}{Application model}  \\ 
 \cmidrule(lr){3-9} && time (m) & rel. & Error(l) & Error(divorce) & MAD(power) & MAD(C)  & Comp(A) \\ \midrule 
\multicolumn{2}{l}{VFI} & 143.024 & 1.000 & 0.127 & 0.000 & 0.019 & 0.882 & 159.057 \\ 
\multicolumn{2}{l}{VFI (no multistart)} & 66.333 & 0.464 & 0.127 & 0.000 & 0.019 & 0.882 & 159.053 \\ 
\multicolumn{2}{l}{EGM (num)} & 15.057 & 0.105 & 0.127 & 0.000 & 0.019 & 0.855 & 158.941 \\ 
\multicolumn{2}{l}{iEGM} \multicolumn{7}{c}{} \\ 
& $\#_C$=25& 2.605 & 0.018 & 0.106 & 0.000 & 0.019 & 2.321 & 168.243 \\ 
& $\#_C$=50& 2.574 & 0.018 & 0.080 & 0.000 & 0.019 & 0.380 & 57.452 \\ 
& $\#_C$=75& 2.818 & 0.020 & 0.050 & 0.000 & 0.019 & 0.382 & 39.878 \\ 
& $\#_C$=100& 3.616 & 0.025 & 0.002 & 0.000 & 0.019 & 0.304 & 35.051 \\ 
& $\#_C$=200& 3.568 & 0.025 & 0.000 & 0.000 & 0.019 & 0.054 & 32.883 \\ 
\bottomrule \end{tabular}
